<a href="https://colab.research.google.com/github/fullstackdata/public2024/blob/main/rag_triad.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install  llama-index-vector-stores-milvus llama-index-llms-huggingface llama-index-embeddings-huggingface sentence-transformers einops


In [ ]:
# Create an index over the documents
from llama_index.core import VectorStoreIndex, StorageContext, load_index_from_storage
from llama_index.vector_stores.milvus import MilvusVectorStore

vector_store = MilvusVectorStore(
    uri="./milvus_llamaindex.db", dim=384, overwrite=False
)
storage_context = StorageContext.from_defaults(vector_store=vector_store)

import logging
import sys

logging.basicConfig(stream=sys.stdout, level=logging.INFO)
logging.getLogger().addHandler(logging.StreamHandler(stream=sys.stdout))

from llama_index.core import VectorStoreIndex, StorageContext
from llama_index.core.schema import TextNode

nodes = [
    TextNode(
        text=(
            "Michael Jordan is a retired professional basketball player,"
            " widely regarded as one of the greatest basketball players of all"
            " time."
        ),
        metadata={
            "category": "Sports",
            "country": "United States",
        },
    ),
    TextNode(
        text=(
            "Angelina Jolie is an American actress, filmmaker, and"
            " humanitarian. She has received numerous awards for her acting"
            " and is known for her philanthropic work."
        ),
        metadata={
            "category": "Entertainment",
            "country": "United States",
        },
    ),
    TextNode(
        text=(
            "Elon Musk is a business magnate, industrial designer, and"
            " engineer. He is the founder, CEO, and lead designer of SpaceX,"
            " Tesla, Inc., Neuralink, and The Boring Company."
        ),
        metadata={
            "category": "Business",
            "country": "United States",
        },
    ),
    TextNode(
        text=(
            "Rihanna is a Barbadian singer, actress, and businesswoman. She"
            " has achieved significant success in the music industry and is"
            " known for her versatile musical style."
        ),
        metadata={
            "category": "Music",
            "country": "Barbados",
        },
    ),
    TextNode(
        text=(
            "Cristiano Ronaldo is a Portuguese professional footballer who is"
            " considered one of the greatest football players of all time. He"
            " has won numerous awards and set multiple records during his"
            " career."
        ),
        metadata={
            "category": "Sports",
            "country": "Portugal",
        },
    ),
]



from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# quantization_config = BitsAndBytesConfig(load_in_8bit=True)
# model_name = "NousResearch/DeepHermes-3-Llama-3-8B-Preview"
# model_name = "microsoft/phi-4"
model_name = "NousResearch/DeepHermes-3-Llama-3-8B-Preview"
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    # quantization_config=quantization_config
)

tokenizer = AutoTokenizer.from_pretrained(model_name, device_map='auto' )

from llama_index.core.llms import ChatMessage
from llama_index.llms.huggingface import HuggingFaceLLM

llm = HuggingFaceLLM(tokenizer=tokenizer, model=model, device_map='auto')

from llama_index.core import Settings
Settings.llm = llm

from llama_index.embeddings.huggingface import HuggingFaceEmbedding

embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")
Settings.embed_model = embed_model

index = VectorStoreIndex(nodes, storage_context=storage_context)





/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the cpu.


In [ ]:
response = index.as_query_engine().query("Tell me about Christian Ronaldo.")
print(response)


 Christian Ronaldo is a professional footballer from Portugal who is widely regarded as one of the greatest football players of all time. He has achieved numerous awards and set multiple records throughout his illustrious career.


In [ ]:
import warnings
import os

warnings.filterwarnings('ignore')

In [ ]:
!pip install trulens-apps-llamaindex


In [ ]:
query_engine = index.as_query_engine()

In [ ]:
from trulens.apps.llamaindex import TruLlama

tru_query_engine_recorder = TruLlama(query_engine)

with tru_query_engine_recorder as recording:
    print(query_engine.query("What all companies did Elon Musk found?"))

ModuleNotFoundError: No module named 'trulens'